# Item 05 — Gênero estrutural: features novas, sonda do canal e re-treino

Gênero deixou de ser uma tabela aprendida de 530×32 parâmetros livres e passou a ser descrito
por atributos com fórmula, derivados da rede gênero↔gênero do MGD+
([ADR-0003](../docs/adr/0003-atributos-de-genero-derivados.md)). Este notebook roda no Colab e faz três coisas:

1. **mostra o que as features novas são** — nos dois regimes de split, com a fórmula à vista;
2. **mede se o canal de gênero conduz** (item 21) — camada a camada, com e sem `cooccurs`,
   comparando os atributos novos contra o ruído N(0; 0,1) que a tabela antiga era na prática;
3. **re-treina** a config vencedora da qualificação sobre o grafo reconstruído e compara o
   `val_mse` com o do grafo antigo (0,000749).

O que o item 04 já mediu ([`docs/diagnostico-ablacao.md`](../docs/diagnostico-ablacao.md)): com o modelo antigo,
esvaziar as 9.866 arestas gênero↔gênero **não mudava nenhum embedding de música** além de 3e−08.
Por isso a seção 2 vem antes do treino: se o canal continuar inerte, atributos novos não movem número.

## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + os artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.

In [1]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")

✓ Colab pronto. cwd = /content/music-influence-gnn
⚠️ FALTAM no repositório: ['data/processed/graph/hetero_full_current.pt', 'data/processed/graph/hetero_full_pre_pandemia.pt']


In [2]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
torch.manual_seed(42); np.random.seed(42)

from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.graph.nodes import GENRE_FEATURE_NAMES
from music_diffusion_gnn.graph.temporal import mask_until
from music_diffusion_gnn.models.encoder import HeteroSpatialEncoder
from music_diffusion_gnn.models.diffusion_gnn import MusicDiffusionGNN
from music_diffusion_gnn.training.dataset import (
    SPLIT_REGIMES, aggregate_weekly, temporal_split, build_samples, build_pop_bank,
)
from music_diffusion_gnn.training.trainer import Config, train_one, evaluate

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IS_COLAB and DEVICE != "cuda":
    raise RuntimeError("GPU não ativada no Colab: Ambiente de execução → Alterar tipo de runtime → GPU.")
print("ROOT =", ROOT, "| DEVICE =", DEVICE)

ImportError: cannot import name 'graph_path' from 'music_diffusion_gnn.graph.build' (/content/music-influence-gnn/src/music_diffusion_gnn/graph/build.py)

In [ ]:
# Artefatos DURÁVEIS no Drive (sobrevivem à desconexão do Colab)
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/music-influence-gnn/item05_genero_estrutural")
else:
    OUT = ROOT / "results" / "item05_genero_estrutural"
OUT.mkdir(parents=True, exist_ok=True)
print("artefatos →", OUT)

## 1. O que as features novas são

Cada gênero passa a ter quatro colunas, computadas **só sobre os anos inteiramente contidos na
janela de treino** do regime (`current` → 2017–2019, `pre_pandemia` → 2017–2018):

| coluna | fórmula |
|---|---|
| `degree` | número de gêneros distintos com que coocorre (self-loops descartados, anos agregados antes da contagem) |
| `weighted_degree` | soma dos pesos de coocorrência dessas arestas |
| `n_artists` | artistas distintos com a tag do gênero nos arquivos anuais de artista |
| `absent_from_network` | 1 quando o gênero não tem nenhuma aresta nos anos de treino |

As três primeiras entram como `log1p` seguido de escore-z; `Avg_Popularity` e `Avg_Streams`
**não são lidas** em nenhum ponto (são agregados de todo o período e vazam o alvo).

In [ ]:
grafos = {r: torch.load(graph_path(r, GRAPH_DIR), weights_only=False) for r in SPLIT_REGIMES}

for r, g in grafos.items():
    x = g["genre"].x
    print(f"{r:14s} x_genre={tuple(x.shape)}  train_years={g['genre'].train_years}  "
          f"absent_from_network={100 * x[:, 3].mean():.1f}%")
assert all(list(g["genre"].feature_names) == GENRE_FEATURE_NAMES for g in grafos.values())

In [ ]:
# Os atributos crus (antes do log1p/escore-z), para leitura humana
from music_diffusion_gnn.data.loaders import load_artists_years, load_genre_network
from music_diffusion_gnn.graph.nodes import genre_attributes

crus = {r: genre_attributes(load_genre_network(reg.train_years),
                            load_artists_years(reg.train_years))
        for r, reg in SPLIT_REGIMES.items()}

display(crus["current"].sort_values("weighted_degree", ascending=False).head(10))

In [ ]:
# Os dois regimes lado a lado, mesmos gêneros: 2019 entra em um e não no outro
lado_a_lado = pd.concat({r: d["weighted_degree"] for r, d in crus.items()}, axis=1).fillna(0)
top = crus["current"].sort_values("weighted_degree", ascending=False).head(15).index
display(lado_a_lado.loc[top])

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(top))
ax.barh(y - 0.2, lado_a_lado.loc[top, "current"], height=0.4, label="current (2017-2019)", color="#2a78d6")
ax.barh(y + 0.2, lado_a_lado.loc[top, "pre_pandemia"], height=0.4, label="pre_pandemia (2017-2018)", color="#eb6834")
ax.set_yticks(y, top); ax.invert_yaxis(); ax.set_xlabel("grau ponderado"); ax.legend()
fig.tight_layout(); plt.show()

## 2. O canal de gênero conduz? (item 21)

O item 04 mediu que, com o modelo treinado da qualificação, esvaziar `cooccurs` não alterava
nenhum embedding de música além de 3e−08. A pergunta desta seção é onde o sinal morre e se a
representação nova muda isso.

A sonda roda o encoder **camada a camada**, com pesos apenas inicializados (seed fixa: mede a
arquitetura, não o que foi aprendido), em duas montagens de `x_genre`: os atributos novos e um
ruído N(0; 0,1) de 32 dimensões, que é o que a tabela antiga era na prática — ela recebia
gradiente só pelo caminho gênero→artista→música e ficava perto da inicialização.

Com `layers = 3` o caminho existe: `cooccurs` (camada 1) → `rev_has_genre` (camada 2) →
`performs` (camada 3). A medição diz quanto dele chega.

In [ ]:
SEMANA_SONDA = 208   # primeira semana do span de teste no regime current

def sonda_canal_genero(g, x_genre, semana=SEMANA_SONDA, encoder=None, hidden=128, layers=3, seed=0):
    """Roda o encoder camada a camada com e sem `cooccurs` e devolve as métricas por camada."""
    snap = mask_until(g, semana)
    if encoder is None:
        torch.manual_seed(seed)
        encoder = HeteroSpatialEncoder(g.metadata(), hidden=hidden, layers=layers, dropout=0.0)
    encoder = encoder.to(DEVICE).eval()

    def rodar(sem_cooccurs):
        x = {k: v.to(DEVICE) for k, v in snap.x_dict.items()}
        x["genre"] = x_genre.to(DEVICE)
        ei = {k: v.to(DEVICE) for k, v in snap.edge_index_dict.items()}
        if sem_cooccurs:
            ei[("genre", "cooccurs", "genre")] = torch.zeros((2, 0), dtype=torch.long, device=DEVICE)
        saidas, h = [], x
        with torch.no_grad():
            for conv in encoder.convs:
                h = {k: v.relu() for k, v in conv(h, ei).items()}
                saidas.append(h)
        return saidas

    completo, ablado = rodar(False), rodar(True)
    linhas = []
    for li, (a, b) in enumerate(zip(completo, ablado), 1):
        for nt in ("genre", "artist", "music"):
            d = (a[nt] - b[nt]).abs()
            linhas.append({
                "camada": li, "tipo": nt,
                "l2_medio": a[nt].norm(dim=1).mean().item(),
                "frac_zeros": (a[nt] == 0).float().mean().item(),
                "dif_media": d.mean().item(), "dif_max": d.max().item(),
                "frac_nos_alterados": (d.max(dim=1).values > 1e-6).float().mean().item(),
            })
    return pd.DataFrame(linhas)


g_cur = grafos["current"]
x_ruido = torch.empty(g_cur["genre"].num_nodes, 32).normal_(0, 0.1)   # a tabela antiga, na prática

sonda = pd.concat({
    "atributos (ADR-0003)": sonda_canal_genero(g_cur, g_cur["genre"].x),
    "ruido 32d (antes)":    sonda_canal_genero(g_cur, x_ruido),
}, names=["x_genre"]).reset_index(level=0)
display(sonda[sonda["tipo"] == "music"])
sonda.to_parquet(OUT / "sonda_canal_genero_pesos_iniciais.parquet", index=False)
display(sonda)

**Como ler.** A linha que decide é `tipo = music` na camada 3: é o embedding que a cabeça
temporal consome. `dif_media` é quanto o embedding de música muda quando `cooccurs` é esvaziado —
a mesma quantidade que o item 04 mediu em 3e−08 com o modelo treinado.

Nas camadas 1 e 2 a diferença em `music` é zero **por construção**: o caminho gênero→música só se
fecha na terceira camada. Se `dif_media` na camada 3 for da ordem do ruído de float, o canal não
conduz nem com features novas, e o item 05 muda de escopo (gênero sai do grafo, ou o encoder muda).

## 3. Re-treino sobre o grafo reconstruído

Config vencedora da qualificação (`W=12, hidden=128, layers=3, lr=5e-4`), agora sobre o grafo com
gênero estrutural. A referência a bater é o `val_mse` do grafo antigo: **0,000749**.

Cada seed grava um checkpoint no Drive assim que termina, e a célula pula seeds já feitas
(retomada após desconexão).

In [ ]:
REGIME = "current"          # troque para "pre_pandemia" e rode de novo
SEEDS  = [42]               # a Phase 6 pede 3 seeds: [42, 43, 44]
VAL_MSE_GRAFO_ANTIGO = 0.000749

g = grafos[REGIME].to(DEVICE)
ts = pd.read_parquet(ROOT / "data" / "processed" / "timeseries.parquet")
weekly = aggregate_weekly(ts)
splits_df = temporal_split(weekly, regime=REGIME)
pop_bank = build_pop_bank(weekly, NMAP_PATH, n_music=g["music"].num_nodes)

fs_global = weekly.groupby(["song_id", "chart"])["week"].min().to_dict()
W = 12
samples = {nome: build_samples(splits_df[nome], W=W, node_id_map_path=NMAP_PATH, first_seen=fs_global)
           for nome in ("train", "val", "test")}
print({k: len(v) for k, v in samples.items()})

In [ ]:
resultados = []
for seed in SEEDS:
    ckpt = OUT / f"gnn_{REGIME}_seed{seed}.pt"
    if ckpt.exists():
        r = torch.load(ckpt, map_location="cpu", weights_only=False)
        print(f"seed {seed}: já treinada (val_mse={r['val_mse']:.6f}) — pulando")
        resultados.append({"seed": seed, "val_mse": r["val_mse"], "curva": r.get("val_curve")})
        continue

    cfg = Config(W=W, hidden=128, layers=3, lr=5e-4, seed=seed)
    r = train_one(cfg, {"train": samples["train"], "val": samples["val"]}, g,
                  device=DEVICE, pop_bank=pop_bank)
    torch.save({"config_str": str(cfg), "W": cfg.W, "hidden": cfg.hidden, "layers": cfg.layers,
                "lr": cfg.lr, "dropout": cfg.dropout, "seed": seed, "split_regime": REGIME,
                "val_mse": r.val_mse, "val_curve": r.val_curve, "train_curve": r.train_curve,
                "state_dict": {k: v.cpu() for k, v in r.best_state_dict.items()}}, ckpt)
    print(f"seed {seed}: val_mse={r.val_mse:.6f}  params={r.n_params}  t={r.elapsed_sec/60:.1f}min → {ckpt.name}")
    resultados.append({"seed": seed, "val_mse": r.val_mse, "curva": r.val_curve})

tabela = pd.DataFrame([{"seed": r["seed"], "val_mse": r["val_mse"]} for r in resultados])
tabela["vs_grafo_antigo"] = tabela["val_mse"] - VAL_MSE_GRAFO_ANTIGO
display(tabela)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for r in resultados:
    if r.get("curva"):
        ax.plot(r["curva"], label=f"seed {r['seed']}")
ax.axhline(VAL_MSE_GRAFO_ANTIGO, ls="--", c="#888", label="grafo antigo (0,000749)")
ax.set_xlabel("época"); ax.set_ylabel("val MSE"); ax.set_yscale("log"); ax.legend()
fig.tight_layout(); plt.show()

## 4. A sonda de novo, agora com os pesos treinados

Esta é a medição que decide o item 21: a seção 2 mostra se a arquitetura conduz, esta mostra se o
modelo **treinado** usa o canal. Se `dif_media` em `music` continuar na ordem de 1e−08, o
treino zerou o canal de novo e gênero não está contribuindo, por melhores que sejam os atributos.

In [ ]:
ck = torch.load(OUT / f"gnn_{REGIME}_seed{SEEDS[0]}.pt", map_location="cpu", weights_only=False)
modelo = MusicDiffusionGNN(g.metadata(), hidden=ck["hidden"], layers=ck["layers"],
                           dropout=ck["dropout"], pop_bank=pop_bank)
sd = dict(ck["state_dict"]); sd.pop("pop_bank", None)
faltando, sobrando = modelo.load_state_dict(sd, strict=False)
assert set(faltando) <= {"pop_bank"} and not sobrando, (faltando, sobrando)

# O treino injeta a popularidade da semana como feature de música (R1); a sonda usa a mesma
# montagem, senão o encoder recebe menos colunas do que viu no treino.
g_sonda = g.clone()
g_sonda["music"].x = torch.cat([g["music"].x, pop_bank[SEMANA_SONDA].to(g["music"].x.device)], dim=1)

sonda_treinada = sonda_canal_genero(g_sonda, g["genre"].x, encoder=modelo.encoder)
sonda_treinada.to_parquet(OUT / f"sonda_canal_genero_treinada_{REGIME}.parquet", index=False)
display(sonda_treinada)

## 5. Avaliação rápida contra a persistência

Fecha o ciclo com os números de val e test held-out do modelo re-treinado. A comparação completa
(escada de cinco modelos, dois recortes, três horizontes) é o item 10; aqui é só o sinal de que o
grafo reconstruído não quebrou nada.

In [ ]:
modelo = modelo.to(DEVICE).eval()
fc_val = evaluate(model=modelo, splits={"train": samples["train"], "val": samples["val"]},
                  weekly_df=weekly, val_split_df=splits_df["val"], g=g,
                  mode="forecasting", max_cotraj_edges=None, device=DEVICE)
fc_test = evaluate(model=modelo, splits={"train": samples["train"], "val": samples["test"]},
                   weekly_df=weekly, val_split_df=splits_df["test"], g=g,
                   mode="forecasting", max_cotraj_edges=None, device=DEVICE)

resumo = pd.DataFrame([
    {"split": s, "regime_chart": c,
     "gnn_rmse": d[f"rmse_{c}"], "gnn_mse": d[f"mse_{c}"], "persist_mse": d[f"persist_mse_{c}"]}
    for s, d in (("val", fc_val), ("test", fc_test)) for c in ("viral50", "top200")
])
resumo["gnn_vence"] = resumo["gnn_mse"] < resumo["persist_mse"]
resumo.to_parquet(OUT / f"avaliacao_{REGIME}.parquet", index=False)
display(resumo)

## 6. O que levar de volta para o repositório

Do Drive (`item05_genero_estrutural/`):

- `sonda_canal_genero_pesos_iniciais.parquet` e `sonda_canal_genero_treinada_{regime}.parquet` — item 21
- `gnn_{regime}_seed*.pt` — checkpoints do grafo reconstruído (os anteriores a ADR-0003 não carregam mais)
- `avaliacao_{regime}.parquet` — val/test contra persistência

Depois: rodar de novo com `REGIME = "pre_pandemia"`, e com `SEEDS = [42, 43, 44]` quando a
escada da Phase 6 começar.